In [7]:
import os
import yaml
from pathlib import Path

# Find the project root by walking up from the notebook's folder until config.yml turns up
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config.yml').exists())

with open(ROOT / 'config.yml') as f:
    cfg = yaml.safe_load(f)

data_root = cfg['environments'][cfg['environment']]['data_root']

def d(subpath):
    return os.path.join(data_root, subpath)

In [ ]:
# !pip install earthengine-api --quiet   # uncomment if needed

import ee
import geopandas as gpd

ee.Authenticate()
ee.Initialize(project='ee-<your_id>')  



Earth Engine initialised.


In [ ]:
# Region asset 
# Uses the full region boundary to export one raster covering the entire area.
#Read local file and reproject to WGS84

gdf = gpd.read_file(d(cfg['data']['plotting']['study_zones'])).to_crs("EPSG:4326")
region_fc = ee.FeatureCollection(gdf.__geo_interface__)
region_geom = region_fc.geometry()

# Years to run - change for each year you want it for
YEARS = [2023] # 2023, 2024, 2025


# We use a spring melt window instead of a full water year to reduce compute.
# Adjust these dates for your region's typical melt season.
wyStartDay    = '-02-01'   # start of primary search window 
wyEndDay      = '-08-01'   # end of primary search window

# NDSI_threshold from Crumley et al 2020
NDSI_threshold = 15

# low_value 
low_value = 0.07

# prior_days
prior_days = 5

# minNoSnowdays 
minNoSnowdays = 5

# projection 
projection = 'EPSG:4326'

# Export folder --- this sends to gdrive
imagesFolder = 'SDDperYear'   

#sanity sheck
print('Comfirm user inputs')
print(f'  NDSI_threshold : {NDSI_threshold}')
print(f'  low_value      : {low_value}')
print(f'  prior_days     : {prior_days}')
print(f'  minNoSnowdays  : {minNoSnowdays}')
print(f'  Window         : {wyStartDay} → {wyEndDay} + 30 adv. days')

User inputs set.
  NDSI_threshold : 15
  low_value      : 0.07
  prior_days     : 5
  minNoSnowdays  : 5
  Window         : -02-01 → -08-01 + 30 adv. days


In [ ]:
bounds_info = region_fc.geometry().bounds().getInfo()
coords      = bounds_info['coordinates'][0]   # outer ring of bbox

lons = [c[0] for c in coords]
lats = [c[1] for c in coords]

xmin, xmax = min(lons), max(lons)
ymin, ymax = min(lats), max(lats)

# Make sure to have a concrete client-side geometry which is safe to pass to Export.image.toDrive
region_geom = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])

#print just for sanity
print(f'Region loaded.')
print(f'  Lon: {xmin:.4f} → {xmax:.4f}')
print(f'  Lat: {ymin:.4f} → {ymax:.4f}')




Region loaded.
  Lon: -128.9259 → -120.0013
  Lat: 56.0898 → 60.0048
Region loaded.


sanity checks

In [9]:
print(region_fc.size().getInfo())        # should be > 0
print(region_geom.bounds().getInfo())    # should show real coordinates, not 0,0

2
{'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-128.92587703972893, 56.08975917007837], [-120.00127162037951, 56.08975917007837], [-120.00127162037951, 60.080037483181655], [-128.92587703972893, 60.080037483181655], [-128.92587703972893, 56.08975917007837]]]}


In [ ]:
##%%
def getMetrics(wyr):
    """
    gets snow disappearance date for each year and returns raster to 
    google drive for the study region
    """

    start     = str(wyr) + wyStartDay
    end       = str(wyr) + wyEndDay
    start_adv = end
    end_adv   = ee.Date(end).advance(30, 'day')

    adv_days = (
        ee.ImageCollection('MODIS/061/MOD10A1')
        .filterDate(start_adv, end_adv)
        .filterBounds(region_geom)
        .size()
        .getInfo()
    )
    print(f'  adv_days = {adv_days}')

    MODIS_cloud_list = (
        ee.ImageCollection('MODIS/061/MOD10A1')
        .filterDate(start, end_adv)
        .filterBounds(region_geom)
        .merge(
            ee.ImageCollection('MODIS/061/MYD10A1')
            .filterDate(start, end_adv)
            .filterBounds(region_geom)
        )
        .sort('system:time_start')
        .map(lambda img: img.expression(
            '(BAND == 250)', {'BAND': img.select('NDSI_Snow_Cover_Class')}
        ))
        .toList(400)
    )

    MODIS_snow_list = (
        ee.ImageCollection('MODIS/061/MOD10A1')
        .filterDate(start, end_adv)
        .filterBounds(region_geom)
        .merge(
            ee.ImageCollection('MODIS/061/MYD10A1')
            .filterDate(start, end_adv)
            .filterBounds(region_geom)
        )
        .sort('system:time_start')
        .map(lambda img: img.expression(
            '(BAND >= threshold) && (BAND <= 100)',
            {'BAND': img.select('NDSI_Snow_Cover'), 'threshold': NDSI_threshold}
        ))
        .toList(400)
    )

    ndays = MODIS_snow_list.length().getInfo() - adv_days
    print(f'  ndays = {ndays}')

    expected      = 365 + (0 if wyr % 4 else 1)
    sddCorrection = expected - ndays if ndays < expected else 0
    print(f'  sddCorrection = {sddCorrection}')

    sddImage               = ee.Image(0)
    future_snow            = ee.Image(0)
    Snow                   = ee.Image(0)
    accSnow                = ee.Image(0)
    ephemeral_counter      = ee.Image(0)
    noSnowCounter          = ee.Image(0)
    minNoSnow              = ee.Image(minNoSnowdays)
    min_sdays_prior_to_SDD = ee.Image(prior_days)
    sddDetected            = ee.Image(0)

    for current_day in range(ndays + adv_days - 1, -1, -1):

        Cloud       = ee.Image(MODIS_cloud_list.get(current_day)).unmask()
        future_snow = Cloud.And(future_snow.Or(Snow))
        Snow        = ee.Image(MODIS_snow_list.get(current_day)).unmask()

        if current_day < ndays:

            snow_occurrence = future_snow.Or(Snow)
            accSnow         = accSnow.add(snow_occurrence)

            min_sdays_prior_to_SDD = min_sdays_prior_to_SDD.where(
                snow_occurrence.eq(0).And(
                    ephemeral_counter.gt(min_sdays_prior_to_SDD)
                ),
                ephemeral_counter
            )

            ephemeral_counter = (
                ephemeral_counter.add(snow_occurrence).multiply(snow_occurrence)
            )

            noSnowCounter = (
                noSnowCounter
                .add(snow_occurrence.eq(0))
                .multiply(sddDetected.eq(0))
            )

            sddDetected = (
                ephemeral_counter.gt(min_sdays_prior_to_SDD)
                .And(noSnowCounter.gt(minNoSnow))
            )

            minNoSnow = minNoSnow.where(sddDetected, noSnowCounter)

            sdd1     = current_day + sddCorrection
            sdd      = min_sdays_prior_to_SDD.add(ee.Image(sdd1))
            sddImage = sddImage.where(sddDetected, sdd)

    sddImage = sddImage.where(accSnow.eq(ndays - 1), ndays + sddCorrection)

    # Water mask 
    # Apply as a mask only; the clip happens once on the final SDD image.
    water = (
        ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
        .select('seasonality')
        .lt(10)     # 1 = not permanent water, 0 = permanent water
        .unmask(1)  # no-data areas treated as not-water
    )

    # SCF image 
    # No clip here  mask only
    scfImage = (
        accSnow
        .divide(ndays)
        .updateMask(accSnow)
        .rename('ccSCF')
        .updateMask(water)
    )

    # No clip here  mask only
    sddImage = (
        sddImage
        .updateMask(sddImage)
        .rename('SDD')
        .updateMask(water)
    )

   
    lower_mask = scfImage.gte(ee.Image(low_value))
    SDD        = sddImage.updateMask(lower_mask)

    # Single clip at the very end on the fully masked image
    return (
    SDD
    .reproject(crs='EPSG:3005', scale=500)  # MODIS sinusoidal
    .clip(region_geom)
)




getMetrics() defined — no water mask, no lower_mask


In [ ]:
##%%
export_tasks = []

for wyr in YEARS:
    print(f'\n{"="*50}')
    print(f'Running getMetrics({wyr})')
    print(f'{"="*50}')

    
    snowReturn = getMetrics(wyr)

    description = f'SDD_{wyr}'
    task = ee.batch.Export.image.toDrive(
        image       = snowReturn,
        description = description,
        folder      = imagesFolder,
        scale       = 500,
        crs         = 'EPSG:4326',
        region      = region_geom,
        maxPixels   = 1e12,
    )
    task.start()
    export_tasks.append((wyr, task))
    print(f'  ✓ Export submitted: {description}  (task {task.id})')

print('\nAll tasks submitted.')
print('Monitor at: https://code.earthengine.google.com/tasks')



Running getMetrics(2023)
  adv_days = 30
  ndays = 370
  sddCorrection = 0
  ✓ Export submitted: SDD_2023  (task WS7CGWGDSECOR27BIWDJPMHY)

All tasks submitted.
Monitor at: https://code.earthengine.google.com/tasks
